In [ ]:
import gymnasium as gym
import numpy as np
import torch
from models import Actor
import os
import matplotlib.pyplot as plt


PATH = "checkpoints/Humanoid-v5/"
ENV = "Humanoid-v5"

In [ ]:
# load actor and enviornment
def load(name:str):
    env = gym.make(ENV, render_mode="human")
    sDim, aDim = env.observation_space.shape[0], env.action_space.shape[0]
    agent = Actor(sDim, aDim)
    agent.load_state_dict(torch.load(os.path.join(PATH, name)))
    return env, agent

# simulate
def run(env, agent, episodes, render):
    for episode in range(episodes):
        state, info = env.reset()
        rewards = 0
        done = False
        while not done:
            action, _ = agent.forward(torch.from_numpy(state).float())
            state, r, terminated, truncated, info = env.step(action.detach().numpy())
            done = terminated or truncated
            rewards += r
            if render is True: 
                env.render()
        print(f"Episode {episode+1}; Rewards: {rewards:.2f}")

In [ ]:
env, agent = load("actor.pth")

run(env, agent, 3, True)

In [ ]:
# test all models save during training

folder = sorted([f for f in os.listdir(PATH) if "actor_" in f])
for f in folder:
        print(f"Model: {f}")
        env, agent = load(f)
        run(env, agent, 1, True)
        env.close()

In [ ]:
# plot rewards

rewards = list(np.load(PATH+"rewards_100k.npy"))
print(len(rewards))
print(rewards[-10:])
steps = np.arange(len(rewards)) * 10

plt.plot(steps, rewards, marker="o", markersize=3, linestyle="-", linewidth=1)

# Labels and title
plt.xlabel("Steps (thousands)")
plt.ylabel("Average Return")
plt.title("SAC Average Rewards per Evaluation")

plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()